In [ ]:
import numpy as np
import pandas as pd
# Regression metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

np.random.seed(17)

## مقدمة



** فحص لوحة المتصدرين ** - إنها تقنية خاصة بالمنافسة ومرتبطة ارتباطًا وثيقًا بتسريب البيانات. هناك نوعان من طرق فحص LB:
* استخراج الحقيقة الأساسية من الجزء العام من LB عن طريق تغيير مجموعة فرعية صغيرة من التنبؤات عند التقديم؛
* استخراج معلومات حول الجزء الخاص من LB عن طريق تقديمها إلى LB العام (استنادًا إلى فئات متسقة).
في هذا البرنامج التعليمي سوف نركز على النوع الأول من التحقيق. في بعض الحالات، يتيح استغلال محدد إمكانية العثور على جميع قيم y لنتيجة لوحة المتصدرين العامة (LB). في حالات أخرى يمكننا الحصول على بعض المعلومات حول توزيع قيم LB y العامة.



## إنشاء وظائف بيئة الاختبار



الآن، سنقوم بإنشاء بعض الوظائف التي ستساعدنا في اختبار برمجيات إكسبلويت.


In [ ]:
def generate_leaderboards(values):
    """
    Generate public and private leaderboard from values.
    """
    df = pd.DataFrame(values, columns=["target"])
    public_lb, private_pb = train_test_split(df, test_size=0.3, shuffle=False)

    return public_lb, private_pb

In [ ]:
def generate_submission(values, leaderboard):
    """
    Generate sample submission from values for leaderboard.
    """
    sample_submission = pd.DataFrame(leaderboard, copy=True)
    sample_submission["target"] = values

    return sample_submission

In [ ]:
def generate_data(values):
    """
    Generate experimental environment: public and private leaderboards, zero sample submissio from values.
    """
    public_lb, private_pb = generate_leaderboards(values)
    zero_submission = generate_submission(0, public_lb)
    n = public_lb.size

    return public_lb, private_pb, zero_submission, n

In [ ]:
def make_submission(predicted_values, metric, leaderboard):
    """
    Evaluate predicted values with metric for leaderboard.
    """
    return metric(leaderboard["target"], predicted_values["target"])


## مقاييس تقييم الانحدار



سنقوم بالتكرار من خلال العديد من مقاييس الانحدار الرئيسية والعثور على بعض نقاط الضعف المتعلقة بكل منها.



### 1. ماي



لنبدأ بمتوسط ​​قياس الخطأ المطلق:



$$\large{MAE(y, \hat{y}) = \frac{\sum_{i=1}^n |y_i - \hat{y}_i|}{n} }$$



أين 
- $y$ - ناقل ذو مكونات حقيقية؛
- $y_i$ - قيمة y الحقيقية؛ 
- $\hat{y}$ - متجه ذو قيم متوقعة (أي مرسلة)؛
- $\hat{y_i}$ - ​​قيمة y المتوقعة؛
- $n$ - عدد نقاط البيانات.



في حالة أن جميع القيم المستهدفة **غير سلبية** يمكننا عدم الإرسال إلى **الحصول على متوسط القيمة المستهدفة**:



$$\large{MAE(y, 0) = \frac{\sum_{i=1}^n |y_i - 0|}{n} = \frac{\sum_{i=1}^n y_i}{n} }$$


In [ ]:
# Generate environment with non negative target values
target_values = np.random.randint(0, 10, size=1000)
public_lb, private_lb, sample_submission, n = generate_data(target_values)

sample_submission.head(3)

In [ ]:
# Make zero submission
p_z = make_submission(sample_submission, mean_absolute_error, public_lb)
print("Zero submission score:", p_z)
print("Public leaderbord target mean:", public_lb["target"].mean())


باستخدام هذه المعلومات يمكننا تعديل توقعاتنا لتحسين درجة LB العامة.



### 2. المشاريع الصغيرة والمتوسطة



تتيح طريقة الفحص التالية **العثور على جميع قيم y** لجميع نقاط البيانات المستخدمة في حساب درجة LB العامة. الآن سنتحدث عن متوسط ​​الخطأ التربيعي:


$$\large{ MSE(y, \hat{y}) = \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{n} }$$



في البداية، لنجعل جميع الأصفار مرسلة ونشير إلى نتيجة النتيجة بـ $p_z$:



$$\large{ p_{z} = \frac{\sum_{i=1}^n (y_i - 0)^2}{n} } = \frac{(y_1 - 0)^2}{n} + \frac{\sum_{i=2}^n (y_i - 0)^2}{n}$$


In [ ]:
# Generate environment
target_values = np.random.randint(-10, 10, size=1000)
public_lb, private_lb, sample_submission, n = generate_data(target_values)

sample_submission.head(3)

In [ ]:
# Make zero submission
p_z = make_submission(sample_submission, mean_squared_error, public_lb)
print("Zero submission score:", p_z)


ستحتوي عمليات الإرسال الأخرى على قيمة واحدة بالضبط تختلف عن صفر إرسال. سنقوم هنا بتغيير قيمة y الأولى من 0 إلى 100:


In [ ]:
# Set first y-value to 100
sample_submission["target"][0] = 100
sample_submission.head(3)


نقوم بإجراء إرسال جديد حيث تم تعيين قيمة y الأولى على 100 ونشير إلى درجة النتيجة كـ $p^1_{100}$:



$$ \large{ p_{100}^{1} = \frac{(y_1 - 100)^2}{n} + \frac{\sum_{i=2}^n (y_i - 0)^2}{n} }$$


In [ ]:
p_1_100 = make_submission(sample_submission, mean_squared_error, public_lb)
print("Submission score:", p_1_100)


الآن، لدينا نظام من المعادلتين الأخيرتين ($p_z$ و$p^1_{100}$) والتي يمكن حلها عبر $y_1$، عن طريق طرح واحدة من الأخرى:



$$\large{ y_1 = \frac{100^2 - n*(p_{100}^1 - p_{z})}{2*100}  }$$


In [ ]:
# Calculate y_1
y1 = (100 ** 2 - n * (p_1_100 - p_z)) / 200
print("Obtained y_1:", y1)
print("Actual y_1:", public_lb["target"][0])


بشكل عام يمكننا الحصول على $y_i$ بهذه الصيغة:



$$\large{ y_i = \frac{100^2 - n*(p_{100}^i - p_{z})}{2*100}  }$$



على سبيل المثال، بالنسبة إلى $y_2$:


In [ ]:
# Set second y-value to 100
sample_submission["target"][0] = 0
sample_submission["target"][1] = 100

# Make submission
p_2_100 = make_submission(sample_submission, mean_squared_error, public_lb)

# Calculate y_2
y2 = (100 ** 2 - n * (p_2_100 - p_z)) / 200
print("Obtained y_2:", y2)
print("Actual y_2:", public_lb["target"][1])


لذلك، في هذه الطريقة يمكننا استخدام مسبار LB واحد بالضبط من أجل الحصول على قيمة y الحقيقية (حتى الدقة العددية) لنقطة بيانات واحدة. تجدر الإشارة إلى أن تطبيق هذه التقنية على متوسط قياس الخطأ المطلق (MAE) وجذر متوسط مربع القياس (RMSE) يجعل من الممكن أيضًا العثور على جميع قيم y لجميع نقاط البيانات المستخدمة في حساب درجة LB العامة



### 3. R تربيع (معامل التحديد)



تتيح لنا الثغرة الأمنية المتعلقة بمقياس $R^2$ العثور على **تباين LB العام** و**العثور على جميع قيم y** لجميع نقاط البيانات المستخدمة في حساب درجة LB العامة. تعريف متري:



$$\large{ R^2(y, \hat{y}) = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y}_i)^2} }$$



يسمى المقام في هذه الصيغة مجموع المربعات، والذي يتناسب مع تباين البيانات.



$$\large{ S_{tot} = \sum_{i=1}^n (y_i - \bar{y}_i)^2 }$$



دعونا نلقي نظرة على صيغة التباين:



$$\large{ Var(y) = \frac{\sum_{i=1}^n (y_i - \bar{y}_i)^2}{n} = \frac{S_{tot}}{n} }$$


لذلك، إذا تمكنا من العثور على $S_{tot}$ في مقياس $R^2$، فيمكننا الحصول على تباين LB العام.



كما هو الحال دائمًا، لنبدأ بصفر إرسال:



$$\large{ p_z = 1 - \frac{\sum_{i=1}^n (y_i - 0)^2}{S_{tot}} = 1 - \frac{y_1^2}{S_{tot}} - \frac{\sum_{i=2}^n (y_i - 0)^2}{S_{tot}} }$$


In [ ]:
## Generate environment
data = np.random.randint(-15, 15, size=1000)
public_lb, private_pb, sample_submission, n = generate_data(data)

sample_submission.head(3)

In [ ]:
# Make zero submission
p_z = make_submission(sample_submission, r2_score, public_lb)
print("Zero submission score:", p_z)


سيكون المسبار الثاني مع ضبط قيمة y الأولى على 100 (ليس هناك مفاجأة هنا):



$$\large{ p^1_{100} = 1 - \frac{(y_1 - 100)^2}{S_{tot}} - \frac{\sum_{i=2}^n (y_i - 0)^2}{S_{tot}} }$$


In [ ]:
# Set first y-value to 100
sample_submission["target"][0] = 100
sample_submission.head(3)

In [ ]:
# Make submission
p_1_100 = make_submission(sample_submission, r2_score, public_lb)
print("Submission score:", p_1_100)


كما في المثال السابق، لدينا الآن نظام من المعادلتين الأخيرتين ($p_z$ و$p^1_{100}$) والتي يمكن حلها عبر $y_1$:



$$\large{ y_1 = \frac{S_{tot}(p^1_{100} - p_z) + 100^2}{2*100} }$$



لكن الشيء الوحيد المتبقي غير المعروف هو $S_{tot}$، والذي يمكن حسابه بإرسال واحد آخر. لذا، سنقوم بتقديم إرسال بقيمة y الأولى مضبوطة على 200.



$$\large{ p^1_{200} = 1 - \frac{(y_1 - 200)^2}{S_{tot}} - \frac{\sum_{i=2}^n (y_i - 0)^2}{S_{tot}} }$$


In [ ]:
# Set first y-value to 200
sample_submission["target"][0] = 200
sample_submission.head(3)

In [ ]:
# Make submission
p_1_200 = make_submission(sample_submission, r2_score, public_lb)
print("Submission score:", p_1_200)


ثم قم بدمج المعادلة مع $p_z$ و$p^1_{200}$ في النظام وحلها عبر $y_1$ (على غرار الخطوة السابقة):



$$\large{ y_1 = \frac{S_{tot}(p^1_{200} - p_z) + 200^2}{2*200} }$$



الآن لدينا معادلتان يمكن حلهما عبر $S_{tot}$، عن طريق استبعاد $y_1$:



$$\large{ S_{tot} = \frac{200^2 - 2*100^2}{2*p^1_{100} - p_z - p^1_{200}} }$$


In [ ]:
# Calculate total sum of squares
s_tot = (200.0 ** 2 - 2 * 100.0 ** 2) / (2 * p_1_100 - p_z - p_1_200)
print("Total sum of squares:", s_tot)


الآن نحن جاهزون لحساب التباين:


In [ ]:
# Calculate variance
var_y = s_tot / n
print("Obtained variance of public LB:", var_y)
print("Actual variance:", public_lb["target"].var(ddof=0))


و $y_1$:


In [ ]:
# Calculate y_1
y1 = (s_tot * (p_1_100 - p_z) + 100 ** 2) / (2 * 100.0)
print("Obtained y_1:", y1)
print("Actual y_1:", public_lb["target"][0])


بعد الآن $S_{tot}$ يمكننا العثور على $y_i$ بهذه الصيغة:



$$\large{ y_i = \frac{S_{tot}(p^i_{100} - p_z) + 100^2}{2*100} }$$



## الخلاصة



لذا فإن الفكرة الأساسية وراء كل الطرق الموصوفة هي تقديم عروض تختلف بقيمة واحدة عن بعضها البعض ثم حل المعادلات (في الغالب عن طريق طرح واحدة من الأخرى).
لا يمكن استخدام المعرفة التي تم الحصول عليها بالطرق الموضحة لتحسين النتيجة في LB الخاص بشكل مباشر، حيث لا يمكننا التحقق من قيم y من هناك. ومع ذلك، إذا جاءت بيانات LB العامة والخاصة من نفس التوزيع، فقد يكون المتوسط ​​أو التباين الذي تم الحصول عليه لـ LB العام مفيدًا.وبطبيعة الحال، فإن الحصول على جميع أساليب قيم y محدود بعدد معين من عمليات الإرسال يوميًا.



## المراجع



1. [متوسط الخطأ المطلق](https://en.wikipedia.org/wiki/Mean_absolute_error)
2. [متوسط الخطأ التربيعي](https://en.wikipedia.org/wiki/Mean_squared_error)
3. [معامل التحديد](https://en.wikipedia.org/wiki/Coefficient_of_determination)
4. [كيفية الحصول على قيم y الدقيقة لجميع نقاط البيانات المستخدمة لحساب نتيجة لوحة المتصدرين العامة في مسابقة "Mercedes-Benz Greener Manufacturing" على Kaggle](https://crowdstats.eu/Kaggle_MercedesBenz_LBprobing.pdf)
5. [كيفية الفوز بمسابقة علوم البيانات: التعلم من أفضل خبراء Kagglers (الأسبوع الثاني)](https://www.coursera.org/learn/competitive-data-science?specialization=aml)
6. [سيناريو "النتيجة المثالية"](https://www.kaggle.com/olegtrott/the-perfect-score-script)